# V38 Step 04 — Merge Category Trips and Combine Milk with Beverages

**Healthy Purchase Cycle | NielsenIQ HMS | 2008–2017**

Six independently prepared category trip files enter this pipeline. The output has **five modeled categories**: FV, combined Beverage, Yogurt, Cereal, and Snack. The separate `milk_y`, milk expenditures, and original beverage outcome/expenditures remain available for audits, but **Milk is not a separate emission**.

**Milk definition:** module 3625 plain refrigerated milk contributes to healthier beverage expenditure; module 3592 flavored refrigerated milk contributes to less-healthy beverage expenditure. These module labels must already have been applied in the upstream Milk extraction and aggregation; this notebook does not recategorize UPCs.

**Key:** `(hhid, panel_year, trip_id, purchase_date)`; preserve identifier strings. The trip universe is the **union of all six category inputs** after restricting to households retained by V38 Step 02. The full-food-spending denominator is constructed separately in Step 05.

**Memory strategy:** DuckDB stages each category into year-partitioned Parquet, then merges and consolidates one calendar year at a time. All data, audit files, and temporary files are written outside GitHub.


In [1]:
# Cell 1 — Libraries and V38-local configuration
# This notebook deliberately preserves the tested DuckDB architecture of
# the supplied six-category merge notebook; there is no pandas full outer join.
# Install into your existing Python environment if needed: %pip install duckdb pandas
from pathlib import Path
import csv
import shutil
import time
import duckdb
import pandas as pd

DATA_ROOT = Path(r"C:/Users/rwen13/Documents/healthy_purchase_data")
V38_NAME = "v38_milk_beverage_combined"
V38_INTERMEDIATE = DATA_ROOT / "intermediate" / V38_NAME

# Outputs are entirely separate from older HMM data products.
OUTPUT_ROOT = V38_INTERMEDIATE / "04_category_trip_merge"
FINAL_DIR = OUTPUT_ROOT / "trip_purchase_panel"
AUDIT_DIR = OUTPUT_ROOT / "validation"
STAGE_ROOT = V38_INTERMEDIATE / "04_category_trip_merge_stage"
TEMP_ROOT = DATA_ROOT / "temp" / V38_NAME / "trip_merge"

# Filter to Step 02 *retained* households (after demographic quality flags).
# The Step 01 eligibility list is intentionally NOT sufficient at this step.
ELIGIBLE_FILE = (V38_INTERMEDIATE / "02_demographics" /
                 "household_demographics_2008_2017.tsv")

YEARS = list(range(2008, 2018))
MEMORY_LIMIT = "4GB"
THREADS = 4

for directory in (FINAL_DIR, AUDIT_DIR, STAGE_ROOT, TEMP_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(database=":memory:")
con.execute(f"SET memory_limit='{MEMORY_LIMIT}'")
con.execute(f"SET threads={THREADS}")
con.execute(f"SET temp_directory='{TEMP_ROOT.as_posix()}'")
print("DuckDB version:", duckdb.__version__)
print("V38 output root:", OUTPUT_ROOT)


DuckDB version: 1.5.5
V38 output root: C:\Users\rwen13\Documents\healthy_purchase_data\intermediate\v38_milk_beverage_combined\04_category_trip_merge


In [2]:
# Cell 2 — Six input datasets, expected schema and conversion helpers
# IMPORTANT: Confirm MILK_FILE is the output from the new plain/flavored milk
# aggregation. If your R aggregation writes to a different path, change ONLY
# MILK_FILE and use the same path in V38 Step 01.
MILK_FILE = (DATA_ROOT / "processed" /
             "milk_trip_level_beverage_classified" /
             "milk_trip_level_2008_2017.tsv")

CATEGORIES = {
    "fv": {
        "file": DATA_ROOT / "processed/fv_trip_level/fv_trip_level_2008_2017.tsv",
        "id_col": "household_code", "outcome": "fv_y",
        "spending": ["fresh_spending", "non_fresh_spending"],
    },
    "milk": {
        "file": MILK_FILE, "id_col": "hhid", "outcome": "milk_y",
        "spending": ["plain_milk_spending", "flavored_milk_spending"],
        # Optional audit fields are preserved if present, and filled with NULL
        # if missing. Optional indicators are NEVER inferred from expenditure.
        "optional": {"total_milk_spending": "DOUBLE",
                     "plain_milk_purchased": "INTEGER",
                     "flavored_milk_purchased": "INTEGER"},
    },
    "yogurt": {
        "file": DATA_ROOT / "processed/yogurt_trip_level/yogurt_trip_level_2008_2017.tsv",
        "id_col": "hhid", "outcome": "yogurt_y",
        "spending": ["skimmed_yogurt_spending", "low_fat_yogurt_spending", "other_yogurt_spending"],
    },
    "cereal": {
        "file": DATA_ROOT / "processed/cereal_trip_level/cereal_trip_level_2008_2017.tsv",
        "id_col": "hhid", "outcome": "cereal_y",
        "spending": ["low_sugar_cereal_spending", "medium_sugar_cereal_spending", "high_sugar_cereal_spending"],
    },
    "snack": {
        "file": DATA_ROOT / "processed/snack_trip_level/snack_trip_level_2008_2017.tsv",
        "id_col": "hhid", "outcome": "snack_y",
        "spending": ["healthy_snack_spending", "unhealthy_snack_spending"],
    },
    "beverage": {
        "file": DATA_ROOT / "intermediate/beverage_trip_level/beverage_trip_level_2008_2017.tsv",
        "id_col": "hhid", "outcome": "beverage_y_original",
        "source_outcome": "beverage_y",
        "spending": ["healthy_beverage_spending_original", "unhealthy_beverage_spending_original"],
        "source_spending": ["healthy_beverage_spending", "unhealthy_beverage_spending"],
    },
}

KEYS = ["hhid", "panel_year", "trip_id", "purchase_date"]
OUTCOMES = [x["outcome"] for x in CATEGORIES.values()]
SPENDING = [v for x in CATEGORIES.values() for v in x["spending"]]
OPTIONAL = CATEGORIES["milk"]["optional"]
ALL_VALUES = OUTCOMES + SPENDING + list(OPTIONAL)
INT_FIELDS = set(OUTCOMES) | {k for k, v in OPTIONAL.items() if v == "INTEGER"}

def sql_path(path):
    return "'" + Path(path).as_posix().replace("'", "''") + "'"

def qcol(col):
    return '"' + col.replace('"', '""') + '"'

def header_columns(path):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        return next(csv.reader(f, delimiter="	"))

source_schema_rows = []
for category, info in CATEGORIES.items():
    path = info["file"]
    if not path.is_file():
        raise FileNotFoundError(f"Missing {category} source: {path}")
    cols = header_columns(path)
    outcome_src = info.get("source_outcome", info["outcome"])
    spending_src = info.get("source_spending", info["spending"])
    required = [info["id_col"], "panel_year", "trip_id", "purchase_date", outcome_src, *spending_src]
    missing = sorted(set(required) - set(cols))
    if missing:
        raise ValueError(f"{category}: missing required columns {missing} in {path}")
    info["columns"] = set(cols)
    source_schema_rows.append({
        "category": category, "source_path": str(path), "n_columns": len(cols),
        "size_gb": round(path.stat().st_size / 1e9, 3),
        "optional_fields_present": ", ".join(sorted(set(OPTIONAL) & set(cols))) if category == "milk" else "",
    })

if not ELIGIBLE_FILE.is_file():
    raise FileNotFoundError(f"Missing V38 Step 02 retained demographics: {ELIGIBLE_FILE}")
if "hhid" not in header_columns(ELIGIBLE_FILE):
    raise ValueError("Step 02 demographics must contain 'hhid'.")

display(pd.DataFrame(source_schema_rows))


,category,source_path,n_columns,size_gb,optional_fields_present
0,fv,C:\Users\rwen13\Documents\healthy_purchase_dat...,7,1.270,
1,milk,C:\Users\rwen13\Documents\healthy_purchase_dat...,12,0.756,"flavored_milk_purchased, plain_milk_purchased,..."
2,yogurt,C:\Users\rwen13\Documents\healthy_purchase_dat...,8,0.282,
3,cereal,C:\Users\rwen13\Documents\healthy_purchase_dat...,8,0.269,
4,snack,C:\Users\rwen13\Documents\healthy_purchase_dat...,7,1.058,
5,beverage,C:\Users\rwen13\Documents\healthy_purchase_dat...,7,1.027,


In [3]:
# Cell 3 — Register the retained demographic household universe
# One row per household, independent of how many demographic years are present.
# Store IDs as VARCHAR to preserve any leading zeros.
con.execute(f"""
    CREATE OR REPLACE TEMP TABLE eligible_hh AS
    SELECT DISTINCT NULLIF(TRIM(CAST(hhid AS VARCHAR)), '') AS hhid
    FROM read_csv({sql_path(ELIGIBLE_FILE)}, delim='\t', header=true, all_varchar=true)
    WHERE hhid IS NOT NULL AND TRIM(hhid) <> ''
""")
n_hh = con.execute("SELECT COUNT(*) FROM eligible_hh").fetchone()[0]
if n_hh == 0:
    raise ValueError("Step 02 retained zero households; cannot merge trips.")
print(f"Eligible households retained after Step 02: {n_hh:,}")


Eligible households retained after Step 02: 80,487


## Stage and validate six independent categories

The six inputs are still independent at the ingestion boundary. Records are checked for valid identifiers, calendar-year alignment, valid outcome codes, and parseable spending **before** filtering to the retained Step 02 households. Each category must have a unique full trip key.

For auditability, the original beverage fields are renamed with `_original` during staging. The milk-specific fields remain separate, and consolidation occurs only after year-level trip union. Missing optional milk indicators remain `NULL` rather than being inferred from dollar amounts.


In [4]:
# Cell 4 — Stage the six datasets using one source at a time
# DuckDB's read_csv(... all_varchar=true) prevents ID type inference.
# Staging is year-partitioned to keep peak memory usage manageable.
stage_audit = []
for category, info in CATEGORIES.items():
    started = time.perf_counter()
    stage_dir = STAGE_ROOT / category
    source = sql_path(info["file"])
    hid = qcol(info["id_col"])
    src_outcome = info.get("source_outcome", info["outcome"])
    src_spending = info.get("source_spending", info["spending"])
    mapped_fields = [(src_outcome, info["outcome"], "INTEGER")]
    mapped_fields += [(raw, alias, "DOUBLE") for raw, alias in zip(src_spending, info["spending"])]
    if category == "milk":
        mapped_fields += [
            (col, col, sql_type)
            for col, sql_type in OPTIONAL.items()
            if col in info["columns"]
        ]
    mapped_aliases = {alias for _, alias, _ in mapped_fields}
    value_sql = ",\n                ".join(
        f"TRY_CAST(NULLIF(TRIM({qcol(raw)}), '') AS {sql_type}) AS {qcol(alias)}"
        for raw, alias, sql_type in mapped_fields
    )
    absent_optional_sql = ",\n                ".join(
        f"CAST(NULL AS {sql_type}) AS {qcol(col)}"
        for col, sql_type in OPTIONAL.items()
        if category == "milk" and col not in mapped_aliases
    )
    extra_sql = f",\n                {absent_optional_sql}" if absent_optional_sql else ""
    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE current_source AS
        SELECT NULLIF(TRIM({hid}), '') AS hhid,
               NULLIF(TRIM(trip_id), '') AS trip_id,
               TRY_CAST(NULLIF(TRIM(purchase_date), '') AS DATE) AS purchase_date,
               TRY_CAST(NULLIF(TRIM(panel_year), '') AS INTEGER) AS source_panel_year,
               {value_sql}{extra_sql}
        FROM read_csv({source}, delim='\t', header=true, all_varchar=true)
    """)
    raw_rows = con.execute("SELECT COUNT(*) FROM current_source").fetchone()[0]
    invalid_keys = con.execute("""
        SELECT COUNT(*) FROM current_source
        WHERE hhid IS NULL OR trip_id IS NULL OR purchase_date IS NULL
           OR EXTRACT(YEAR FROM purchase_date) NOT BETWEEN 2008 AND 2017
           OR source_panel_year IS NULL
           OR source_panel_year <> EXTRACT(YEAR FROM purchase_date)
    """).fetchone()[0]
    if invalid_keys:
        raise ValueError(f"{category}: {invalid_keys:,} invalid keys/dates/source years")

    # Failed numeric casts must never silently turn a purchased product into NULL.
    # Check source values directly, using source column names and expected types.
    checks = [
        f"(NULLIF(TRIM({qcol(raw)}), '') IS NOT NULL AND "
        f"TRY_CAST(NULLIF(TRIM({qcol(raw)}), '') AS {typ}) IS NULL)"
        for raw, _, typ in mapped_fields
    ]
    # This is evaluated against the original input and does not alter the data.
    bad_cast_count = con.execute(f"""
        SELECT COUNT(*) FROM read_csv({source}, delim='\t', header=true, all_varchar=true)
        WHERE {' OR '.join(checks)}
    """).fetchone()[0]
    if bad_cast_count:
        raise ValueError(f"{category}: {bad_cast_count:,} unparseable category values")

    outcome = qcol(info["outcome"])
    bad_outcomes = con.execute(f"""
        SELECT COUNT(*) FROM current_source
        WHERE {outcome} IS NOT NULL AND {outcome} NOT IN
             ({'1, 2, 3' if category == 'cereal' else '1, 2'})
    """).fetchone()[0]
    if bad_outcomes:
        raise ValueError(
            f"{category}: {bad_outcomes:,} unexpected source outcomes. "
            "Category-only input should contain 1/2 (cereal 1/2/3), not zero."
        )
    null_outcomes = con.execute(f"SELECT COUNT(*) FROM current_source WHERE {outcome} IS NULL").fetchone()[0]
    if null_outcomes:
        raise ValueError(f"{category}: {null_outcomes:,} source category trips have NULL outcome")

    # Validate the full trip key before any category-level consolidation.
    duplicate_keys = con.execute("""
        SELECT COUNT(*) FROM (
            SELECT hhid, trip_id, purchase_date,
                   EXTRACT(YEAR FROM purchase_date) AS panel_year
            FROM current_source GROUP BY ALL HAVING COUNT(*) > 1
        )
    """).fetchone()[0]
    if duplicate_keys:
        raise ValueError(f"{category}: {duplicate_keys:,} duplicate trip keys")

    if category == "milk":
        # Total milk spending is an optional redundant audit field; its value
        # should match the sum of mutually exclusive plain/flavored amounts.
        if "total_milk_spending" in info["columns"]:
            discrepancies = con.execute("""
                SELECT COUNT(*) FROM current_source
                WHERE total_milk_spending IS NOT NULL
                  AND (plain_milk_spending IS NULL OR flavored_milk_spending IS NULL
                    OR ABS(total_milk_spending - plain_milk_spending - flavored_milk_spending) > 0.005)
            """).fetchone()[0]
            if discrepancies:
                raise ValueError(f"Milk: {discrepancies:,} rows fail spending reconciliation")
        contradictory = con.execute("""
            SELECT COUNT(*) FROM current_source
            WHERE (milk_y = 1 AND plain_milk_purchased = 0)
               OR (milk_y = 2 AND plain_milk_purchased = 1)
               OR (milk_y = 2 AND flavored_milk_purchased = 0)
               OR (plain_milk_purchased IS NOT NULL AND plain_milk_purchased NOT IN (0,1))
               OR (flavored_milk_purchased IS NOT NULL AND flavored_milk_purchased NOT IN (0,1))
        """).fetchone()[0]
        if contradictory:
            raise ValueError(f"Milk: {contradictory:,} inconsistent trip outcome/indicator rows")

    con.execute("""
        CREATE OR REPLACE TEMP VIEW current_eligible AS
        SELECT s.hhid,
               CAST(EXTRACT(YEAR FROM s.purchase_date) AS INTEGER) AS panel_year,
               s.trip_id, s.purchase_date,
               s.* EXCLUDE (hhid, trip_id, purchase_date, source_panel_year)
        FROM current_source s
        INNER JOIN eligible_hh e USING (hhid)
    """)
    kept_rows = con.execute("SELECT COUNT(*) FROM current_eligible").fetchone()[0]
    if stage_dir.exists():
        shutil.rmtree(stage_dir)
    stage_dir.mkdir(parents=True, exist_ok=True)
    if kept_rows:
        con.execute(f"""
            COPY (SELECT * FROM current_eligible)
            TO {sql_path(stage_dir)}
            (FORMAT PARQUET, PARTITION_BY (panel_year), COMPRESSION ZSTD)
        """)
    con.execute("DROP VIEW current_eligible")
    con.execute("DROP TABLE current_source")
    elapsed = time.perf_counter() - started
    stage_audit.append({"category": category, "source_rows": raw_rows,
                        "retained_rows": kept_rows, "duplicate_keys": duplicate_keys,
                        "invalid_key_rows": invalid_keys, "bad_numeric_cast_rows": bad_cast_count,
                        "elapsed_seconds": round(elapsed, 1)})
    print(f"Staged {category:9s}: source={raw_rows:,} retained={kept_rows:,} ({elapsed:.1f}s)")

stage_audit_df = pd.DataFrame(stage_audit)
stage_audit_df.to_csv(AUDIT_DIR / "category_stage_audit.csv", index=False)
display(stage_audit_df)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Staged fv       : source=27,910,490 retained=22,505,343 (11.3s)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

... (5 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Staged beverage : source=22,846,393 retained=18,077,804 (9.9s)


,category,source_rows,retained_rows,duplicate_keys,invalid_key_rows,bad_numeric_cast_rows,elapsed_seconds
0,fv,27910490,22505343,0,0,0,11.3
1,milk,13122809,10455817,0,0,0,6.9
2,yogurt,6049411,4831372,0,0,0,2.8
3,cereal,5729258,4497850,0,0,0,2.7
4,snack,20867975,16710378,0,0,0,8.6
5,beverage,22846393,18077804,0,0,0,9.9


## Yearly trip merge and immediate Milk–Beverage consolidation

All six category records are stacked and grouped by the full trip key. A trip appearing **only in Milk** remains in the final universe. The original beverage outcome is preserved as `beverage_y_original`; the modeled `beverage_y` follows these rules:

- **1:** The trip has a healthy original beverage or plain milk (`beverage_y_original == 1` or `milk_y == 1`).
- **2:** The trip has an original unhealthy beverage or flavored milk but **no** healthy beverage or plain milk.
- **NULL:** Neither category is purchased.

Spending is split by product classification, **not** assigned wholesale according to the trip's priority-coded outcome:

- `healthy_beverage_spending` = original healthy beverage spending + plain milk spending.
- `unhealthy_beverage_spending` = original unhealthy beverage spending + flavored milk spending.

Missing source-category spending is treated as zero only when the *other source category* is purchased. When neither Milk nor Beverage is purchased, the combined spending remains `NULL`; all source audit fields remain unchanged.


In [5]:
# Cell 5 — Merge six categories and construct the five-category panel
merge_audit = []
for year in YEARS:
    started = time.perf_counter()
    projections = []
    per_category_counts = {}
    for category, info in CATEGORIES.items():
        partition = STAGE_ROOT / category / f"panel_year={year}"
        files = sorted(partition.glob("*.parquet")) if partition.exists() else []
        if not files:
            per_category_counts[category] = 0
            continue
        glob_path = sql_path(partition / "*.parquet")
        available = {info["outcome"], *info["spending"]}
        if category == "milk":
            available.update(OPTIONAL)
        projected = []
        for value in ALL_VALUES:
            if value in available:
                projected.append(qcol(value))
            else:
                sql_type = "INTEGER" if value in INT_FIELDS else "DOUBLE"
                projected.append(f"CAST(NULL AS {sql_type}) AS {qcol(value)}")
        projections.append(
            f"SELECT hhid, CAST({year} AS INTEGER) AS panel_year, trip_id, purchase_date, "
            + ", ".join(projected) + f" FROM read_parquet({glob_path})"
        )
        per_category_counts[category] = con.execute(
            f"SELECT COUNT(*) FROM read_parquet({glob_path})"
        ).fetchone()[0]

    if not projections:
        print(f"Skipping {year}: no retained category trips")
        continue
    stacked = " UNION ALL ".join(projections)
    values_sql = ", ".join(f"MAX({qcol(c)}) AS {qcol(c)}" for c in ALL_VALUES)
    raw_merge = f"""
        SELECT hhid, panel_year, trip_id, purchase_date, {values_sql}
        FROM ({stacked}) stacked
        GROUP BY hhid, panel_year, trip_id, purchase_date
    """
    # Distinguish nonpurchase NULL from a purchased less-healthy alternative (2).
    # 1 wins when healthy and unhealthy drinks/milk are on the same trip.
    # A source category value of 0 is not expected in the classified trip inputs.
    combined_sql = f"""
        SELECT m.*,
            CASE WHEN beverage_y_original = 1 OR milk_y = 1 THEN 1
                 WHEN beverage_y_original = 2 OR milk_y = 2 THEN 2
                 ELSE NULL END AS beverage_y,
            CASE WHEN beverage_y_original IS NOT NULL OR milk_y IS NOT NULL
                 THEN COALESCE(healthy_beverage_spending_original, 0.0)
                    + COALESCE(plain_milk_spending, 0.0)
                 ELSE NULL END AS healthy_beverage_spending,
            CASE WHEN beverage_y_original IS NOT NULL OR milk_y IS NOT NULL
                 THEN COALESCE(unhealthy_beverage_spending_original, 0.0)
                    + COALESCE(flavored_milk_spending, 0.0)
                 ELSE NULL END AS unhealthy_beverage_spending
        FROM ({raw_merge}) m
    """
    year_dir = FINAL_DIR / f"panel_year={year}"
    year_dir.mkdir(parents=True, exist_ok=True)
    target = year_dir / "trips.parquet"
    con.execute(f"COPY ({combined_sql}) TO {sql_path(target)} (FORMAT PARQUET, COMPRESSION ZSTD)")
    ref = f"read_parquet({sql_path(target)})"
    out_count, milk_only, bev_only, joint = con.execute(f"""
        SELECT COUNT(*),
            COUNT(*) FILTER (WHERE milk_y IS NOT NULL AND beverage_y_original IS NULL),
            COUNT(*) FILTER (WHERE milk_y IS NULL AND beverage_y_original IS NOT NULL),
            COUNT(*) FILTER (WHERE milk_y IS NOT NULL AND beverage_y_original IS NOT NULL)
        FROM {ref}
    """).fetchone()
    if out_count > sum(per_category_counts.values()):
        raise AssertionError("Merged trips exceed stacked source-category rows")
    # Every eligible Milk source trip must be present after union, including milk-only trips.
    if milk_only + joint != per_category_counts.get("milk", 0):
        raise AssertionError(f"{year}: Milk trips lost during union/consolidation")
    if bev_only + joint != per_category_counts.get("beverage", 0):
        raise AssertionError(f"{year}: Original Beverage trips lost during union/consolidation")
    merge_audit.append({
        "panel_year": year, "unique_merged_trips": out_count,
        "source_category_rows_sum": sum(per_category_counts.values()),
        "milk_only_trips": milk_only, "beverage_only_trips": bev_only,
        "both_milk_and_beverage_trips": joint,
        **{f"{cat}_trips": per_category_counts.get(cat, 0) for cat in CATEGORIES},
        "elapsed_seconds": round(time.perf_counter() - started, 1),
    })
    print(f"{year}: {out_count:,} merged trips; {milk_only:,} milk-only -> {target}")

merge_audit_df = pd.DataFrame(merge_audit)
merge_audit_df.to_csv(AUDIT_DIR / "five_category_merge_summary.csv", index=False)
display(merge_audit_df)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2008: 3,368,393 merged trips; 549,019 milk-only -> C:\Users\rwen13\Documents\healthy_purchase_data\intermediate\v38_milk_beverage_combined\04_category_trip_merge\trip_purchase_panel\panel_year=2008\trips.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2009: 3,699,872 merged trips; 592,893 milk-only -> C:\Users\rwen13\Documents\healthy_purchase_data\intermediate\v38_milk_beverage_combined\04_category_trip_merge\trip_purchase_panel\panel_year=2009\trips.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2010: 4,027,911 merged trips; 632,381 milk-only -> C:\Users\rwen13\Documents\healthy_purchase_data\intermediate\v38_milk_beverage_combined\04_category_trip_merge\trip_purchase_panel\panel_year=2010\trips.parquet


... (11 outputs left out)

2016: 4,122,803 merged trips; 515,196 milk-only -> C:\Users\rwen13\Documents\healthy_purchase_data\intermediate\v38_milk_beverage_combined\04_category_trip_merge\trip_purchase_panel\panel_year=2016\trips.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2017: 3,704,316 merged trips; 455,912 milk-only -> C:\Users\rwen13\Documents\healthy_purchase_data\intermediate\v38_milk_beverage_combined\04_category_trip_merge\trip_purchase_panel\panel_year=2017\trips.parquet


,panel_year,unique_merged_trips,source_category_rows_sum,milk_only_trips,beverage_only_trips,both_milk_and_beverage_trips,fv_trips,milk_trips,yogurt_trips,cereal_trips,snack_trips,beverage_trips,elapsed_seconds
0,2008,3368393,6536410,549019,1095144,462999,1774793,1012018,366907,431564,1392985,1558143,3.0
1,2009,3699872,7240379,592893,1186272,504057,2014098,1096950,421809,476576,1540617,1690329,3.3
2,2010,4027911,7912872,632381,1291150,537880,2223099,1170261,488421,512687,1689374,1829030,3.8
3,2011,4206761,8240867,648341,1339873,534614,2365763,1182955,521357,506999,1789306,1874487,4.1
4,2012,4154835,8047689,610407,1376001,499047,2328160,1109454,506294,481560,1747173,1875048,4.2
5,2013,4119659,7948447,580378,1385640,475124,2319665,1055502,518270,457145,1737101,1860764,4.4
6,2014,4196848,8087130,563081,1437248,464550,2395754,1027631,529172,451859,1780916,1901798,4.6
7,2015,4145029,8018361,535236,1438970,459703,2426893,994939,524254,432819,1740783,1898673,4.9
8,2016,4122803,7947185,515196,1457986,448249,2431165,963445,507036,408621,1730683,1906235,4.9
9,2017,3704316,7099224,455912,1296547,386750,2225953,842662,447852,338020,1561440,1683297,4.6


In [6]:
# Cell 6 — Outcome, key and spending identity checks on final Parquet
validation_rows = []
outcome_distributions = []
allowed = {"fv_y": (1,2), "milk_y": (1,2),
           "yogurt_y": (1,2), "cereal_y": (1,2,3),
           "snack_y": (1,2), "beverage_y_original": (1,2),
           "beverage_y": (1,2)}

for year in YEARS:
    target = FINAL_DIR / f"panel_year={year}" / "trips.parquet"
    if not target.is_file():
        continue
    ref = f"read_parquet({sql_path(target)})"
    duplicate_keys = con.execute(f"""
        SELECT COUNT(*) FROM (
            SELECT hhid, panel_year, trip_id, purchase_date FROM {ref}
            GROUP BY ALL HAVING COUNT(*) > 1
        )
    """).fetchone()[0]
    invalid_keys = con.execute(f"""
        SELECT COUNT(*) FROM {ref}
        WHERE hhid IS NULL OR trip_id IS NULL OR purchase_date IS NULL
           OR panel_year <> {year}
    """).fetchone()[0]
    bad_codes = 0
    observed_counts = {}
    for field, possible in allowed.items():
        values = ", ".join(str(v) for v in possible)
        dist = con.execute(f"""
            SELECT CAST({qcol(field)} AS VARCHAR) AS val, COUNT(*) AS n
            FROM {ref} GROUP BY 1 ORDER BY 1
        """).fetchall()
        outcome_distributions.extend(
            {"panel_year": year, "outcome": field,
             "value": "NULL" if val is None else val, "n_trips": n}
            for val, n in dist
        )
        bad, count = con.execute(f"""
            SELECT COUNT(*) FILTER (WHERE {qcol(field)} IS NOT NULL
                                    AND {qcol(field)} NOT IN ({values})),
                   COUNT({qcol(field)})
            FROM {ref}
        """).fetchone()
        bad_codes += bad
        observed_counts[f"{field}_observed"] = count

    # Check both the outcome priority rule and the exact spending sum.
    rule_error, healthy_error, unhealthy_error, milk_total_error = con.execute(f"""
        SELECT
          COUNT(*) FILTER (WHERE beverage_y IS DISTINCT FROM
            (CASE WHEN beverage_y_original=1 OR milk_y=1 THEN 1
                  WHEN beverage_y_original=2 OR milk_y=2 THEN 2 ELSE NULL END)),
          COUNT(*) FILTER (WHERE
            healthy_beverage_spending IS DISTINCT FROM
            (CASE WHEN beverage_y_original IS NOT NULL OR milk_y IS NOT NULL
                  THEN COALESCE(healthy_beverage_spending_original,0.0)
                       + COALESCE(plain_milk_spending,0.0) ELSE NULL END)),
          COUNT(*) FILTER (WHERE
            unhealthy_beverage_spending IS DISTINCT FROM
            (CASE WHEN beverage_y_original IS NOT NULL OR milk_y IS NOT NULL
                  THEN COALESCE(unhealthy_beverage_spending_original,0.0)
                       + COALESCE(flavored_milk_spending,0.0) ELSE NULL END)),
          COUNT(*) FILTER (WHERE total_milk_spending IS NOT NULL AND
             ABS(total_milk_spending - COALESCE(plain_milk_spending, 0.0)
                                     - COALESCE(flavored_milk_spending,0.0)) > 0.005)
        FROM {ref}
    """).fetchone()
    validation_rows.append({
        "panel_year": year, "duplicate_trip_keys": duplicate_keys,
        "invalid_keys_or_year": invalid_keys, "invalid_outcome_codes": bad_codes,
        "beverage_priority_errors": rule_error,
        "healthy_beverage_spend_errors": healthy_error,
        "unhealthy_beverage_spend_errors": unhealthy_error,
        "milk_spending_reconciliation_errors": milk_total_error,
        **observed_counts,
    })

validation_df = pd.DataFrame(validation_rows)
distribution_df = pd.DataFrame(outcome_distributions)
validation_df.to_csv(AUDIT_DIR / "five_category_merge_validation.csv", index=False)
distribution_df.to_csv(AUDIT_DIR / "five_category_outcome_distribution.csv", index=False)
display(validation_df)
if not distribution_df.empty:
    display(distribution_df[distribution_df["panel_year"] == 2008])

error_fields = ["duplicate_trip_keys", "invalid_keys_or_year", "invalid_outcome_codes",
                "beverage_priority_errors", "healthy_beverage_spend_errors",
                "unhealthy_beverage_spend_errors", "milk_spending_reconciliation_errors"]
if not validation_df.empty and validation_df[error_fields].to_numpy().sum() > 0:
    raise AssertionError("V38 trip merge validation failed; inspect audit CSV files")
print("PASS: V38 trip keys, five-category outcomes and beverage-spending identities")
print("NOTE: NULL outcomes mean category nonpurchase; they are NOT recoded to zero")


,panel_year,duplicate_trip_keys,invalid_keys_or_year,invalid_outcome_codes,beverage_priority_errors,healthy_beverage_spend_errors,unhealthy_beverage_spend_errors,milk_spending_reconciliation_errors,fv_y_observed,milk_y_observed,yogurt_y_observed,cereal_y_observed,snack_y_observed,beverage_y_original_observed,beverage_y_observed
0,2008,0,0,0,0,0,0,0,1774793,1012018,366907,431564,1392985,1558143,2107162
1,2009,0,0,0,0,0,0,0,2014098,1096950,421809,476576,1540617,1690329,2283222
2,2010,0,0,0,0,0,0,0,2223099,1170261,488421,512687,1689374,1829030,2461411
3,2011,0,0,0,0,0,0,0,2365763,1182955,521357,506999,1789306,1874487,2522828
4,2012,0,0,0,0,0,0,0,2328160,1109454,506294,481560,1747173,1875048,2485455
5,2013,0,0,0,0,0,0,0,2319665,1055502,518270,457145,1737101,1860764,2441142
6,2014,0,0,0,0,0,0,0,2395754,1027631,529172,451859,1780916,1901798,2464879
7,2015,0,0,0,0,0,0,0,2426893,994939,524254,432819,1740783,1898673,2433909
8,2016,0,0,0,0,0,0,0,2431165,963445,507036,408621,1730683,1906235,2421431
9,2017,0,0,0,0,0,0,0,2225953,842662,447852,338020,1561440,1683297,2139209


,panel_year,outcome,value,n_trips
0,2008,fv_y,1,1286953
1,2008,fv_y,2,487840
2,2008,fv_y,NULL,1593600
3,2008,milk_y,1,984550
4,2008,milk_y,2,27468
5,2008,milk_y,NULL,2356375
6,2008,yogurt_y,1,186617
7,2008,yogurt_y,2,180290
8,2008,yogurt_y,NULL,3001486
9,2008,cereal_y,1,52916


PASS: V38 trip keys, five-category outcomes and beverage-spending identities
NOTE: NULL outcomes mean category nonpurchase; they are NOT recoded to zero


### Outputs and downstream contract

- **Parquet:** `healthy_purchase_data/intermediate/v38_milk_beverage_combined/04_category_trip_merge/trip_purchase_panel/panel_year=YYYY/trips.parquet`.
- **Audits:** `.../04_category_trip_merge/validation/` with staging, merge, outcome, and spending checks.
- **Five modeled choices:** `fv_y`, **combined `beverage_y`**, `yogurt_y`, `cereal_y`, `snack_y`.
- **Combined beverage spending:** `healthy_beverage_spending` and `unhealthy_beverage_spending`.
- **Audit-only Milk/original Beverage:** `milk_y`, `plain_milk_spending`, `flavored_milk_spending`, optional `total_milk_spending` and purchase flags, `beverage_y_original`, `healthy_beverage_spending_original`, `unhealthy_beverage_spending_original`.

This notebook **does not join demographics or food spending**, and does not expand to shopping trips outside the six observed category files. The next step computes the food-spending denominator without double counting either Milk or Beverage. `NULL` outcomes mark category nonpurchase; they are not unhealthy alternatives.

If the actual V38 milk output path differs from the configured path, update `MILK_FILE` here **and in Step 01** before running. All outputs remain outside the GitHub repository.
